In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import Image, display
import time

def scrape_nate_news(section_name, section_code):
    """
    Nate 뉴스 섹션별 스크래핑 함수

    Args:
        section_name: 섹션 이름 (예: '최신뉴스', '정치')
        section_code: 섹션 코드 (예: 'n0100', 'n0200')
    """
    base_url = 'https://news.nate.com'
    url = f'{base_url}/recent?mid={section_code}'

    print(f"\n[{section_name}] 섹션 뉴스 수집 시작")

    try:
        headers = {
            'User-Agent': (
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/91.0.4472.124 Safari/537.36'
            )
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'lxml')
        news_items = soup.select('div.mlt01')

        if not news_items:
            print(f"{section_name} 섹션에서 뉴스를 찾을 수 없습니다.")
            return

        print(f"총 {len(news_items)}개의 뉴스 발견")

        for idx, item in enumerate(news_items[:10], 1):
            print(f"\n뉴스 {idx}")

            link_tag = item.select_one('a')
            article_link = ''
            if link_tag:
                article_link = link_tag.get('href', '')
                if article_link:
                    article_link = urljoin(base_url, article_link)

            img_tag = item.select_one('img')
            if img_tag and img_tag.get('src'):
                img_src = img_tag.get('src')
                img_url = urljoin(base_url, img_src)

                if img_url.startswith('//'):
                    img_url = 'https:' + img_url

                print(f"이미지 URL: {img_url}")

                try:
                    img_response = requests.get(img_url, timeout=5)
                    if img_response.status_code == 200:
                        display(Image(img_response.content, width=300))
                except Exception as e:
                    print(f"이미지 로드 실패: {e}")
            else:
                print("이미지 없음")

            title_tag = item.select_one('h2.tit, h4.tit, h2, h4')
            if title_tag:
                title = title_tag.get_text(strip=True)
            elif link_tag:
                title = link_tag.get_text(strip=True)
            else:
                title = '제목을 찾을 수 없음'

            print(f"제목: {title}")

            if article_link:
                print(f"링크: {article_link}")
            else:
                print("링크 없음")

        print(f"[{section_name}] 섹션 수집 완료")

    except requests.exceptions.RequestException as e:
        print(f"요청 오류: {e}")
    except Exception as e:
        print(f"처리 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()


sections = [
    ('최신뉴스', 'n0100'),
    ('정치', 'n0200'),
    ('경제', 'n0300'),
    ('사회', 'n0400'),
    ('세계', 'n0500'),
    ('IT/과학', 'n0600')
]

print("Nate 뉴스 스크래핑 시작")

for section_name, section_code in sections:
    scrape_nate_news(section_name, section_code)
    time.sleep(1)

print("모든 섹션 스크래핑 완료")


In [ ]:
import requests
from bs4 import BeautifulSoup
import os

def download_one_episode(title, no, url):
    # 요청 헤더 설정
    headers = {
        'referer': url,
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36'
    }

    # 웹페이지 요청
    res = requests.get(url, headers=headers)
    if not res.ok:
        print(f'페이지 요청 실패: {res.status_code}')
        return

    # HTML 파싱
    soup = BeautifulSoup(res.text, 'html.parser')

    # 웹툰 이미지 태그 선택
    img_tags = soup.select("img[src*='IMAG01']")
    print('이미지 개수:', len(img_tags))
    if not img_tags:
        print('이미지를 찾을 수 없습니다.')
        return

    # 이미지 URL 리스트 생성
    img_url_list = [img['src'] for img in img_tags]

    # 저장할 디렉토리 생성
    base_dir = os.path.join('img', title, str(no))
    os.makedirs(base_dir, exist_ok=True)

    # 이미지 다운로드
    for img_url in img_url_list:
        img_res = requests.get(img_url, headers=headers)
        if not img_res.ok:
            continue

        img_data = img_res.content
        file_path = os.path.join(base_dir, os.path.basename(img_url))

        with open(file_path, 'wb') as f:
            f.write(img_data)
            print(f'Downloaded: {file_path} ({len(img_data):,} bytes)')

    print(f'\n "{title}" {no}화 이미지 다운로드 완료!')


download_one_episode(
    '일렉시드',
    341,
    'https://comic.naver.com/webtoon/detail?titleId=717481&no=341&week=wed'
)


이미지 개수: 88
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_1.jpg (87,143 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_2.jpg (256,127 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_3.jpg (184,536 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_4.jpg (182,867 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_5.jpg (112,615 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_6.jpg (169,889 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_7.jpg (157,876 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_8.jpg (181,837 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_9.jpg (203,632 bytes)
Downloaded: img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68

In [ ]:
import os
import requests
from bs4 import BeautifulSoup

def download_one_episode(title, no, url):
    headers = {
        'user-agent': 'Mozilla/5.0',
        'referer': url
    }

    res = requests.get(url, headers=headers)
    if not res.ok:
        print('회차 페이지 요청 실패')
        return

    soup = BeautifulSoup(res.text, 'html.parser')

    # PC 웹툰 이미지 selector
    img_tags = soup.select('img[src*="IMAG01"]')
    print('이미지 개수:', len(img_tags))

    if not img_tags:
        print('이미지를 찾을 수 없습니다.')
        return

    save_dir = f'img/{title}/{no}'
    os.makedirs(save_dir, exist_ok=True)

    for idx, img in enumerate(img_tags, start=1):
        img_url = img['src']
        img_res = requests.get(img_url, headers=headers)

        file_path = f'{save_dir}/{idx}.jpg'
        with open(file_path, 'wb') as f:
            f.write(img_res.content)

    print(f' {no}화 다운로드 완료')


download_one_episode(
    '배달왕',
    130,
    'https://comic.naver.com/webtoon/detail?titleId=823933&no=130'
)




이미지 개수: 44
✅ 130화 다운로드 완료
